# Fine-tuning BLIP — descrição de condição do livro

Antes de rodar:

1. **Add Input** (barra lateral) → anexar o Kaggle Dataset das fotos:
   `pedrocanoas/fotos-livros`. O código vem direto do GitHub (célula 1),
   não precisa mais de um segundo Dataset.
2. **Settings** → Accelerator: **GPU (T4 x2 ou P100)** · Internet: **On**
   (necessário tanto pro `git clone` quanto pra baixar os pesos do BLIP do
   Hugging Face na primeira execução)

> **Nota sobre o layout do Kaggle:** o Kaggle monta os datasets em
> `/kaggle/input/datasets/<usuário>/<slug>/...` (não mais direto em
> `/kaggle/input/<slug>/...`). Confirme rodando `!find /kaggle/input -maxdepth 4`
> se os caminhos abaixo não baterem com a estrutura do seu notebook.

In [ ]:
# Célula 1 — clona o código direto do GitHub (sempre pega a versão mais recente do main)
!git clone --depth 1 https://github.com/Pedrocanoas/tcc.git /kaggle/working/repo
%cd /kaggle/working/repo/model

In [ ]:
# Célula 2 — instala as deps que faltam (não reinstala torch: o Kaggle já vem com build CUDA pronta)
!pip install -q -r requirements-kaggle.txt
# o Kaggle traz um torchao antigo (0.10); o peft (LoRA do BLIP-2) aborta ao encontrar versão
# abaixo da 0.16, mesmo sem usar — nada aqui depende de torchao, então sai
!pip uninstall -y -q torchao

In [ ]:
# Célula 3 — aponta os diretórios de dados/checkpoints para /kaggle/working (única área gravável)
import os

os.environ["TCC_DATASET_RAW_DIR"] = "/kaggle/input/datasets/pedrocanoas/fotos-livros"
os.environ["TCC_DATASET_PROCESSED_DIR"] = "/kaggle/working/processed"
os.environ["TCC_CHECKPOINTS_DIR"] = "/kaggle/working/checkpoints"

In [ ]:
# Célula 4 — comparação com 3 conjuntos (treino 80% / validação 10% / teste 10%, por livro).
# Todos os modelos: mesma base limpa (códigos internos removidos, coleções fora do treino),
# mesma semente (42), no máximo 20 epochs com parada antecipada (3 epochs sem melhorar a
# perda de validação), melhor checkpoint escolhido pela validação. O teste nunca é usado
# pra escolher nada — só pra reportar os números finais.
#
# Duas execuções (o Kaggle limita cada uma a 12 h): GROUP = "A" e depois GROUP = "B".
# Saída de cada modelo em /kaggle/working/results/<nome>/:
#   train.log, history.json (perda por epoch), eval_val.log, eval_test.log
# (o BLIP sem fine-tuning, no grupo B, só tem os dois logs de avaliação).
# Só o checkpoint do BLIP 384 congelado é mantido (candidato a produção).
import shutil
import subprocess
from pathlib import Path

GROUP = "B"
RAW = "/kaggle/input/datasets/pedrocanoas/fotos-livros"
COMMON = f"--raw-dir {RAW} --epochs 20 --patience 3 --seed 42 --max-length 128 --exclude-colecoes"
BLIP_RUNS = {
    "A": {
        "blip_384_congelado": "--batch-size 8",
        "blip_384_2camadas": "--batch-size 8 --unfreeze-vision-layers 2 --vision-lr 1e-5",
    },
    "B": {
        "blip_576_congelado": "--batch-size 8 --image-size 576",
        # batch 8 não coube na T4 (CUDA out of memory): batch 4 x 2 acumulações = mesmo batch efetivo
        "blip_576_2camadas": "--batch-size 4 --grad-accum 2 --image-size 576 --unfreeze-vision-layers 2 --vision-lr 1e-5",
    },
}
RESULTS = Path("/kaggle/working/results")


def sh(cmd: str) -> None:
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=False)


for name, extra in BLIP_RUNS[GROUP].items():
    out, ckpt = RESULTS / name, Path(f"/kaggle/working/ckpt_{name}")
    out.mkdir(parents=True, exist_ok=True)
    sh(f"python kaggle_train.py {COMMON} {extra} --checkpoints-dir {ckpt} > {out}/train.log 2>&1")
    sh(f"grep -E '== epoch|parada|Error' {out}/train.log")
    shutil.copy(ckpt / "history.json", out / "history.json")
    for split in ("val", "test"):
        sh(f"python -u -m src.evaluate --checkpoint {ckpt}/best --split {split} > {out}/eval_{split}.log 2>&1")
        sh(f"tail -3 {out}/eval_{split}.log")
    shutil.rmtree(ckpt / "last", ignore_errors=True)
    if name != "blip_384_congelado":
        shutil.rmtree(ckpt, ignore_errors=True)

if GROUP == "A":
    out = RESULTS / "blip2_opt27b"
    out.mkdir(parents=True, exist_ok=True)
    sh(f"python -u -m src.blip2_experiment --epochs 20 --patience 3 --seed 42 --batch-size 4 --output-dir {out} > {out}/train.log 2>&1")
    sh(f"grep -E 'versões|treináveis|== |parada|Error' {out}/train.log")
    (out / "best_trainable.pt").unlink(missing_ok=True)

if GROUP == "B":
    # linha de base: BLIP original sem fine-tuning (checkpoint inexistente -> src.evaluate usa o modelo base)
    out = RESULTS / "blip_sem_finetuning"
    out.mkdir(parents=True, exist_ok=True)
    for split in ("val", "test"):
        sh(f"python -u -m src.evaluate --checkpoint /kaggle/working/nao_existe --split {split} > {out}/eval_{split}.log 2>&1")
        sh(f"tail -3 {out}/eval_{split}.log")

sh(f"ls -la {RESULTS}/*")

In [ ]:
# Célula 5 (opcional) — sanity check: gera uma legenda de exemplo com o checkpoint treinado
import glob

sample_photos = sorted(glob.glob("/kaggle/input/datasets/pedrocanoas/fotos-livros/livro001/foto*.jpg"))[:1]
if sample_photos and Path("/kaggle/working/ckpt_blip_384_congelado/best").exists():
    !python -m src.infer --checkpoint /kaggle/working/ckpt_blip_384_congelado/best {sample_photos[0]}

In [ ]:
# Célula 6 — tamanho do que vai pro Output (os checkpoints "last" e os dos experimentos já
# foram apagados na célula 4; só fica o best do BLIP 384 congelado)
!du -sh /kaggle/working/results /kaggle/working/ckpt_* 2>/dev/null

Depois de rodar tudo, clique em **Save Version** (Save & Run All) para commitar o
notebook — a pasta `results/` (logs, `history.json`, avaliações) e o checkpoint
`ckpt_blip_384_congelado/best/` aparecem na aba **Output**, ou via Kaggle API:

```sh
kaggle kernels output <seu-usuario>/<slug-do-notebook> -p ./kaggle-output
```

Copie a pasta `best/` escolhida para `model/checkpoints/best/` (caminho que
`infer.py` usa por padrão localmente).